## Notebook — Sync Divalto UNIFIÉ → Supabase (`divalto_mouvements_all`)

### Objectif
Alimenter UNE table Supabase unique `divalto_mouvements_all`, **sans filtre de préfixe** et **au grain ligne analytique**, à partir de `lakehouse_gold.mouv_gold` (+ `C8_gold` compta).

Remplace à terme les exports filtrés `be_divalto_mouvements` (NASKEO only) et `it_divalto_*` (fournisseur only). Tous les modules budgétaires (BE / IT / SPV) filtrent ensuite ce dont ils ont besoin via des vues SQL.

### Pourquoi
- L'export BE filtrait `CCN/CFN/FCN/FFN` (NASKEO) → les pièces TerGreen (CF*/FF*/CC*/FC*), dont les affaires montage `M`, étaient perdues.
- L'export BE agrégeait par `(numero_piece, source)` en gardant `first(axe_0001)` → les pièces multi-affaires perdaient toutes leurs lignes sauf une.
- L'export IT ne prenait que le fournisseur (CF*/FF*) → aucun CA client.

### Principe de cette version
- **Aucun filtre de préfixe.** Toutes les pièces commande (FULLCDNO) et facture (FULLFANO) sont exportées.
- **Aucune agrégation par pièce.** Chaque ligne analytique de `mouv_gold` devient une ligne Supabase → multi-affaires préservé.
- **`line_uid`** = hash SHA-256 du contenu de la ligne → clé d'idempotence (relançable sans doublon).
- **`montant_ht`** = montant signé brut (convention `sens=2 → négatif`). Neutre : chaque module applique sa propre lecture vente/achat via le préfixe.

### Envoi
Edge Function `bulk-upsert`, `conflict_key = line_uid`, mode `upsert`, batches de 500.

In [ ]:
# ─── 0) Config ───────────────────────────────────────────
# Credentials : utilisées telles quelles si injectées par le pipeline
# (Base parameters) ; sinon récupérées depuis la variableLibrary
# (exécution manuelle). Auto-fallback → fonctionne dans les deux modes.
try:
    SUPABASE_URL  # définie ? (mode pipeline)
except NameError:
    env = notebookutils.variableLibrary.getLibrary("env-temp")
    SUPABASE_URL      = env.SUPABASE_URL
    SYNC_SECRET       = env.SYNC_SECRET
    SUPABASE_ANON_KEY = env.SUPABASE_PUBLISHABLE_KEY

import math, time, re, requests
from decimal import Decimal
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, DateType, IntegerType

GOLD_DB         = "lakehouse_gold"
BULK_UPSERT_URL = f"{SUPABASE_URL.rstrip('/')}/functions/v1/bulk-upsert"
BATCH_SIZE      = 500
SLEEP_S         = 0.02
TABLE           = "divalto_mouvements_all"
CONFLICT_KEY    = "line_uid"

def bulk_headers():
    return {
        "Content-Type":  "application/json",
        "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
        "apikey":        SUPABASE_ANON_KEY,
        "x-sync-secret": SYNC_SECRET,
    }

print("[0] Config OK —", SUPABASE_URL)

In [ ]:
# ─── 1) Helpers sérialisation + envoi (identiques aux autres notebooks) ───
_CTRL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")
def _clean_str(s):
    if s is None: return None
    return _CTRL_RE.sub("", str(s)).rstrip() or None
def _to_jsonable(v):
    if v is None: return None
    if isinstance(v, bool): return v
    if isinstance(v, int): return v
    if isinstance(v, float): return None if (math.isnan(v) or math.isinf(v)) else round(v, 2)
    if isinstance(v, Decimal):
        try:
            f = float(v); return None if (math.isnan(f) or math.isinf(f)) else round(f, 2)
        except Exception: return _clean_str(str(v))
    if isinstance(v, (date, datetime)): return v.isoformat()
    if isinstance(v, (bytes, bytearray)): return _clean_str(v.decode("utf-8", errors="ignore"))
    if isinstance(v, str):
        s = _clean_str(v)
        if not s or s in ("1899-12-30", "0000-00-00"): return None
        return s
    if isinstance(v, dict): return {k: _to_jsonable(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)): return [_to_jsonable(x) for x in v]
    return _clean_str(str(v))
def _sanitize(rec): return {k: _to_jsonable(v) for k, v in rec.items()}
def _post_batch(records):
    if not records: return 0, None
    payload = {"table": TABLE, "conflict_key": CONFLICT_KEY, "records": [_sanitize(r) for r in records], "on_conflict": "upsert"}
    try:
        r = requests.post(BULK_UPSERT_URL, headers=bulk_headers(), json=payload, timeout=180)
        if r.status_code in (200, 201, 204): return len(records), None
        return 0, f"HTTP {r.status_code}: {r.text[:300]}"
    except Exception as e:
        return 0, str(e)
def send_to_supabase(records, label):
    ok_total, err_total, errs = 0, 0, []
    for i in range(0, len(records), BATCH_SIZE):
        chunk = records[i:i+BATCH_SIZE]
        nb, err = _post_batch(chunk)
        ok_total += nb
        if err: err_total += len(chunk); errs.append(f"batch {i//BATCH_SIZE+1}: {err}")
        time.sleep(SLEEP_S)
    print(f"[{label}] OK {ok_total}/{len(records)} | ERR {err_total}")
    for m in errs: print("  ⚠️", m)
    return ok_total, err_total
print("[1] Helpers OK")

In [ ]:
# ─── 2) mouv_gold (aucun filtre de préfixe) ───────────────
df = spark.table(f"{GOLD_DB}.mouv_gold")
print(f"[2] mouv_gold : {df.count()} lignes brutes")

In [ ]:
# ─── 3) COMMANDES — grain ligne, tous préfixes ────────────
df_cmd = (
    df.filter(F.col("FULLCDNO").isNotNull() & (F.trim(F.col("FULLCDNO")) != ""))
      .select(
        F.lit("commande").alias("doc_type"),
        F.trim(F.col("FULLCDNO").cast(StringType())).alias("numero_piece"),
        F.col("prefcdno").cast(StringType()).alias("prefix"),
        F.col("sens").cast(IntegerType()).alias("sens"),
        F.trim(F.col("axe_0001").cast(StringType())).alias("axe_0001"),
        F.trim(F.col("axe_0002").cast(StringType())).alias("axe_0002"),
        F.when(F.col("sens") == 2, -F.col("mont")).otherwise(F.col("mont")).cast(DoubleType()).alias("montant_ht"),
        F.trim(F.col("tiers").cast(StringType())).alias("tiers_code"),
        F.trim(F.col("tiers_fou").cast(StringType())).alias("nom_tiers"),
        F.lit(None).cast(StringType()).alias("fullcdno_lie"),
        F.col("cddt").cast(DateType()).alias("date_piece"),
        F.year(F.col("cddt")).cast(IntegerType()).alias("exercice"),
        F.trim(F.col("projet").cast(StringType())).alias("projet"),
        F.col("dos").cast(StringType()).alias("dos"),
        F.trim(F.col("des").cast(StringType())).alias("libelle"),
        F.lit("gescom").alias("source"),
        F.lit("MOUV_GOLD").alias("source_system"),
      )
)
print(f"[3] Lignes commandes : {df_cmd.count()}")

In [ ]:
# ─── 4) FACTURES gescom — grain ligne, tous préfixes ──────
df_fac = (
    df.filter(F.col("FULLFANO").isNotNull() & (F.trim(F.col("FULLFANO")) != ""))
      .select(
        F.lit("facture").alias("doc_type"),
        F.trim(F.col("FULLFANO").cast(StringType())).alias("numero_piece"),
        F.col("preffano").cast(StringType()).alias("prefix"),
        F.col("sens").cast(IntegerType()).alias("sens"),
        F.trim(F.col("axe_0001").cast(StringType())).alias("axe_0001"),
        F.trim(F.col("axe_0002").cast(StringType())).alias("axe_0002"),
        F.when(F.col("sens") == 2, -F.col("mont")).otherwise(F.col("mont")).cast(DoubleType()).alias("montant_ht"),
        F.trim(F.col("tiers").cast(StringType())).alias("tiers_code"),
        F.trim(F.col("tiers_fou").cast(StringType())).alias("nom_tiers"),
        F.trim(F.col("FULLCDNO").cast(StringType())).alias("fullcdno_lie"),
        F.col("fadt").cast(DateType()).alias("date_piece"),
        F.year(F.col("fadt")).cast(IntegerType()).alias("exercice"),
        F.trim(F.col("projet").cast(StringType())).alias("projet"),
        F.col("dos").cast(StringType()).alias("dos"),
        F.trim(F.col("des").cast(StringType())).alias("libelle"),
        F.lit("gescom").alias("source"),
        F.lit("MOUV_GOLD").alias("source_system"),
      )
)
print(f"[4] Lignes factures gescom : {df_fac.count()}")

In [ ]:
# ─── 5) Union + line_uid (hash de contenu) ────────────────
df_all = df_cmd.unionByName(df_fac)

df_all = df_all.withColumn(
    "line_uid",
    F.sha2(F.concat_ws("|",
        F.coalesce(F.col("doc_type"), F.lit("")),
        F.coalesce(F.col("numero_piece"), F.lit("")),
        F.coalesce(F.col("prefix"), F.lit("")),
        F.coalesce(F.col("axe_0001"), F.lit("")),
        F.coalesce(F.col("axe_0002"), F.lit("")),
        F.coalesce(F.col("montant_ht").cast(StringType()), F.lit("")),
        F.coalesce(F.col("sens").cast(StringType()), F.lit("")),
        F.coalesce(F.col("date_piece").cast(StringType()), F.lit("")),
        F.coalesce(F.col("dos"), F.lit("")),
        F.coalesce(F.col("libelle"), F.lit("")),
    ), 256)
)
# Dédoublonne d'éventuelles lignes strictement identiques (même hash)
df_all = df_all.dropDuplicates(["line_uid"])
print(f"[5] Total lignes à pousser : {df_all.count()}")
df_all.groupBy("doc_type", "prefix").count().orderBy("doc_type", "prefix").show(50, truncate=False)

In [ ]:
# ─── 6) Envoi vers Supabase ───────────────────────────────
records = [row.asDict(recursive=True) for row in df_all.collect()]
ok, err = send_to_supabase(records, "DIVALTO_ALL")
print("=" * 50)
print(f"Pièces envoyées : {ok} | erreurs : {err}")
print("=" * 50)